In [1]:
# Cell 1: Setup and Mount Google Drive
from google.colab import drive
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define project paths
project_path = '/content/drive/MyDrive/customer-intelligence-pipeline'
raw_data_path = os.path.join(project_path, 'data/raw')

print(f"✅ Project path: {project_path}")
print(f"✅ Data will be saved to: {raw_data_path}")



Mounted at /content/drive
✅ Project path: /content/drive/MyDrive/customer-intelligence-pipeline
✅ Data will be saved to: /content/drive/MyDrive/customer-intelligence-pipeline/data/raw


In [3]:
# Cell 2: Generate 1000 Realistic Customers
def generate_customers(n=1000):
    """
    Generate synthetic customer data with realistic patterns

    Args:
        n: Number of customers to generate

    Returns:
        DataFrame with customer information
    """
    np.random.seed(42)  # For reproducibility
    random.seed(42)

    # Generate unique customer IDs
    customer_ids = [f'CUST_{i:05d}' for i in range(1, n+1)]

    # Generate signup dates (spread over last 2 years)
    end_date = datetime.now()
    start_date = end_date - timedelta(days=730)
    signup_dates = [start_date + timedelta(days=random.randint(0, 730)) for _ in range(n)]

    # Customer segments (realistic distribution)
    segments = np.random.choice(['Standard', 'Premium', 'Enterprise'], n, p=[0.6, 0.3, 0.1])

    # Geographic regions
    regions = np.random.choice(['North America', 'Europe', 'Asia', 'South America'], n)

    # Email subscription preference
    email_subscribed = np.random.choice([True, False], n, p=[0.7, 0.3])

    # Create DataFrame
    data = {
        'customer_id': customer_ids,
        'signup_date': signup_dates,
        'segment': segments,
        'region': regions,
        'email_subscribed': email_subscribed
    }

    return pd.DataFrame(data)

# Generate customers
df_customers = generate_customers(1000)

print(f"✅ Generated {len(df_customers)} customers")
print(f"\nFirst 5 customers:")
print(df_customers.head())
print(f"\nData shape: {df_customers.shape}")
print(f"\nData types:\n{df_customers.dtypes}")



✅ Generated 1000 customers

First 5 customers:
  customer_id                signup_date     segment         region  \
0  CUST_00001 2025-09-13 15:49:50.749659    Standard           Asia   
1  CUST_00002 2024-03-22 15:49:50.749659  Enterprise  South America   
2  CUST_00003 2023-12-24 15:49:50.749659     Premium         Europe   
3  CUST_00004 2024-09-05 15:49:50.749659    Standard  South America   
4  CUST_00005 2024-08-05 15:49:50.749659    Standard  South America   

   email_subscribed  
0              True  
1              True  
2              True  
3              True  
4              True  

Data shape: (1000, 5)

Data types:
customer_id                 object
signup_date         datetime64[ns]
segment                     object
region                      object
email_subscribed              bool
dtype: object


In [4]:
# Cell 3: Generate 5000 Realistic Orders
def generate_orders(customers_df, n_orders=5000):
    """
    Generate synthetic order data linked to customers

    Args:
        customers_df: DataFrame with customers
        n_orders: Number of orders to generate

    Returns:
        DataFrame with order information
    """
    np.random.seed(42)
    random.seed(42)

    customer_ids = customers_df['customer_id'].values
    signup_dates = dict(zip(customers_df['customer_id'], customers_df['signup_date']))

    order_data = []

    for i in range(n_orders):
        # Pick a random customer
        cust_id = np.random.choice(customer_ids)
        cust_signup = signup_dates[cust_id]

        # Order date must be AFTER customer signup date
        days_active = (datetime.now() - cust_signup).days
        if days_active < 1:
            days_active = 1

        # Random order date between signup and today
        order_date = cust_signup + timedelta(days=random.randint(0, days_active))

        # Order amount (using log-normal distribution for realistic spend patterns)
        # Most customers spend small amounts, some spend large amounts
        amount = round(np.random.lognormal(3, 1) * 10, 2)

        # Order status (most completed, some returned/cancelled)
        status = np.random.choice(['Completed', 'Returned', 'Cancelled'],
                                 p=[0.90, 0.05, 0.05])

        order_data.append({
            'order_id': f'ORD_{i+1:06d}',
            'customer_id': cust_id,
            'order_date': order_date,
            'amount': amount,
            'status': status
        })

    return pd.DataFrame(order_data)

# Generate orders
df_orders = generate_orders(df_customers, 5000)

print(f"✅ Generated {len(df_orders)} orders")
print(f"\nFirst 5 orders:")
print(df_orders.head())
print(f"\nOrder statistics:")
print(df_orders['amount'].describe())
print(f"\nOrder status distribution:")
print(df_orders['status'].value_counts())


✅ Generated 5000 orders

First 5 orders:
     order_id customer_id                 order_date  amount     status
0  ORD_000001  CUST_00103 2025-10-15 15:49:50.749659  115.86  Completed
1  ORD_000002  CUST_00021 2024-04-18 15:49:50.749659  336.30  Completed
2  ORD_000003  CUST_00467 2025-02-04 15:49:50.749659  182.72  Completed
3  ORD_000004  CUST_00872 2025-08-26 15:49:50.749659   79.34  Completed
4  ORD_000005  CUST_00662 2025-11-15 15:49:50.749659  217.55   Returned

Order statistics:
count    5000.000000
mean      341.627398
std       446.960959
min         6.200000
25%       102.122500
50%       203.810000
75%       411.947500
max      9464.630000
Name: amount, dtype: float64

Order status distribution:
status
Completed    4519
Returned      248
Cancelled     233
Name: count, dtype: int64


In [5]:
# Cell 4: Save data to CSV files
# Make sure the directory exists
os.makedirs(raw_data_path, exist_ok=True)

# Save customer data
customers_file = os.path.join(raw_data_path, 'customers.csv')
df_customers.to_csv(customers_file, index=False)

# Save order data
orders_file = os.path.join(raw_data_path, 'orders.csv')
df_orders.to_csv(orders_file, index=False)

print("✅ Successfully saved files to Google Drive:")
print(f"   1. {customers_file}")
print(f"   2. {orders_file}")

# Verify files exist and check sizes
print("\n📁 Files in raw data folder:")
for file in os.listdir(raw_data_path):
    file_path = os.path.join(raw_data_path, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"   ✓ {file} ({size_mb:.2f} MB)")

print("\n✅ Data generation and saving complete!")



✅ Successfully saved files to Google Drive:
   1. /content/drive/MyDrive/customer-intelligence-pipeline/data/raw/customers.csv
   2. /content/drive/MyDrive/customer-intelligence-pipeline/data/raw/orders.csv

📁 Files in raw data folder:
   ✓ customers.csv (0.06 MB)
   ✓ orders.csv (0.31 MB)

✅ Data generation and saving complete!
